In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
!pip install --upgrade "dask[dataframe]"

In [23]:
!pip install --upgrade "dask[distributed]"

In [24]:
import dask.dataframe as dd
import pandas as pd
import numpy as np

from dask.distributed import Client, LocalCluster

# Existe integração com pandas, então customização da configuração do pandas também funciona aqui

# Não limitar a largura das colunas apresentadas
pd.options.display.max_colwidth = None
# Não usar a notação científica (ex: 6.125000e-02) e usar 6 casas decimais (ex: 0.061250)
pd.options.display.float_format = "{:.6f}".format
# Não utilizar matplotlib como engine de gráficos e usar plotly
pd.options.plotting.backend = "plotly"

In [25]:
# Criar cluster local anexado ao kernel do notebook
cluster = LocalCluster()
client = Client(cluster)
client

/usr/local/lib/python3.12/dist-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44049 instead
  warnings.warn(
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:41597
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:44049/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:43901'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:41383'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:42303 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:42303
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:54102
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:41671 name: 1
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:41671
IN

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:44049/status,
Dashboard: http://127.0.0.1:44049/status,Workers: 2
Total threads: 2,Total memory: 12.67 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:41597,Workers: 0
Dashboard: http://127.0.0.1:44049/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:42303,Total threads: 1
Dashboard: http://127.0.0.1:33461/status,Memory: 6.34 GiB
Nanny: tcp://127.0.0.1:43901,


In [26]:
# client.close()
# cluster.close()

## Explorando os datasets

In [42]:
ROOT_DATA_PATH = "drive/MyDrive/data/ml-25m"

In [43]:
movies_df = dd.read_csv(f"{ROOT_DATA_PATH}/movies.csv")
movies_df

,movieId,title,genres
npartitions=1,,,
,int64,string,string
,...,...,...


In [44]:
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [45]:
len(movies_df)

62423

In [46]:
tags_df = dd.read_csv(f"{ROOT_DATA_PATH}/tags.csv")
tags_df

,userId,movieId,tag,timestamp
npartitions=1,,,,
,int64,int64,string,int64
,...,...,...,...


In [47]:
tags_df.head()

,userId,movieId,tag,timestamp
0,3,260,classic,1439472355
1,3,260,sci-fi,1439472256
2,4,1732,dark comedy,1573943598
3,4,1732,great dialogue,1573943604
4,4,7569,so bad it's good,1573943455


In [48]:
len(tags_df)

1093360

In [49]:
# Converter os valores da coluna 'timestamp' de epoch em segundos para datetime
tags_df['timestamp'] = dd.to_datetime(tags_df['timestamp'], unit='s')
tags_df['userId'] = tags_df['userId'].astype(int)
tags_df['movieId'] = tags_df['movieId'].astype(int)
tags_df

,userId,movieId,tag,timestamp
npartitions=1,,,,
,int64,int64,string,datetime64[ns]
,...,...,...,...


In [50]:
tags_df.head()

,userId,movieId,tag,timestamp
0,3,260,classic,2015-08-13 13:25:55
1,3,260,sci-fi,2015-08-13 13:24:16
2,4,1732,dark comedy,2019-11-16 22:33:18
3,4,1732,great dialogue,2019-11-16 22:33:24
4,4,7569,so bad it's good,2019-11-16 22:30:55


In [51]:
ratings_df = dd.read_csv(f"{ROOT_DATA_PATH}/ratings.csv")
ratings_df

,userId,movieId,rating,timestamp
npartitions=10,,,,
,int64,int64,float64,int64
,...,...,...,...
...,...,...,...,...
,...,...,...,...
,...,...,...,...


In [52]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,296,5.000000,1147880044
1,1,306,3.500000,1147868817
2,1,307,5.000000,1147868828
3,1,665,5.000000,1147878820
4,1,899,3.500000,1147868510


In [53]:
len(ratings_df)

25000095

In [54]:
# Converter os valores da coluna 'timestamp' de epoch em segundos para datetime
ratings_df['timestamp'] = dd.to_datetime(ratings_df['timestamp'], unit='s')
ratings_df['userId'] = ratings_df['userId'].astype(int)
ratings_df['movieId'] = ratings_df['movieId'].astype(int)
ratings_df

,userId,movieId,rating,timestamp
npartitions=10,,,,
,int64,int64,float64,datetime64[ns]
,...,...,...,...
...,...,...,...,...
,...,...,...,...
,...,...,...,...


In [55]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,296,5.000000,2006-05-17 15:34:04
1,1,306,3.500000,2006-05-17 12:26:57
2,1,307,5.000000,2006-05-17 12:27:08
3,1,665,5.000000,2006-05-17 15:13:40
4,1,899,3.500000,2006-05-17 12:21:50


In [56]:
dd.read_csv(f"{ROOT_DATA_PATH}/links.csv")

,movieId,imdbId,tmdbId
npartitions=1,,,
,int64,int64,int64
,...,...,...


In [57]:
dd.read_csv(f"{ROOT_DATA_PATH}/links.csv").isnull().sum().compute()

ValueError: Mismatched dtypes found in `pd.read_csv`/`pd.read_table`.

+--------+---------+----------+
| Column | Found   | Expected |
+--------+---------+----------+
| tmdbId | float64 | int64    |
+--------+---------+----------+

Usually this is due to dask's dtype inference failing, and
*may* be fixed by specifying dtypes manually by adding:

dtype={'tmdbId': 'float64'}

to the call to `read_csv`/`read_table`.

Alternatively, provide `assume_missing=True` to interpret
all unspecified integer columns as floats.

In [58]:
# Existem filmes sem valor na coluna 'tmdbId', por esse motivo a coluna não pode ser do tipo int
links_df = dd.read_csv(f'{ROOT_DATA_PATH}/links.csv',
                       dtype={'movieId': int, 'imdbId': int},
                       assume_missing=True)
links_df

,movieId,imdbId,tmdbId
npartitions=1,,,
,int64,int64,float64
,...,...,...


In [59]:
links_df.isnull().sum().compute()

,0
movieId,0
imdbId,0
tmdbId,107


In [60]:
links_df.head()

,movieId,imdbId,tmdbId
0,1,114709,862.000000
1,2,113497,8844.000000
2,3,113228,15602.000000
3,4,114885,31357.000000
4,5,113041,11862.000000


In [61]:
len(links_df)

62423

In [62]:
gtags_df = dd.read_csv(f'{ROOT_DATA_PATH}/genome-tags.csv')
gtags_df

,tagId,tag
npartitions=1,,
,int64,string
,...,...


In [ ]:
gtags_df.head()

In [ ]:
len(gtags_df)

In [ ]:
gscores_df = dd.read_csv(f'{ROOT_DATA_PATH}/genome-scores.csv')
gscores_df

In [ ]:
gscores_df.head()

In [ ]:
len(gscores_df)

## Executando merge de dados

In [ ]:
# Calculando total de linhas
len(movies_df)

In [ ]:
len(links_df)

In [ ]:
movies_df.head()

In [ ]:
links_df.head()

In [ ]:
# Criando operação de merge entre filmes e links
merged_df = dd.merge(movies_df, links_df, on='movieId', how='inner')
merged_df

In [ ]:
merged_df.head()

In [ ]:
len(merged_df)

In [ ]:
# Uma forma alternativa de fazer merge seria utilizando um dos dataframes
movies_df = movies_df.merge(links_df, on='movieId', how='inner')
movies_df.head()

In [ ]:
len(movies_df)

## Quais são os top 10 filmes mais avaliados?

In [ ]:
ratings_df.head()

In [ ]:
type(ratings_df.head())

In [ ]:
type(ratings_df)

In [ ]:
ratings_df

In [ ]:
%%time

len(ratings_df)

In [ ]:
%%time

# Calcular os top 10 filmes com maior quantidade de avaliações
df = ratings_df[['movieId', 'rating']].groupby('movieId')\
  ['rating']\
  .count()\
  .rename('ratings_count')\
  .nlargest(10)\
  .persist() # Para salvar o df computado na memória RAM dos workers. Sai imediatamente.

In [ ]:
%%time

df.max().compute() # Esse comando aguarda o cáculo da célula anterior terminar

In [ ]:
%%time

df.max().compute() # Uma vez persistido em RAM o mesmo comando é mais rápido

In [ ]:
%%time

df.compute()

In [ ]:
movies_df[['movieId', 'title']].compute()

In [ ]:
df.compute()\
  .reset_index()\
  .merge(
      movies_df[['movieId', 'title']].compute(), # Atenção, essa linha pode estourar a memória do client
      on='movieId',
      how='inner'
  )\
  .set_index('movieId')

In [ ]:
df.compute().reset_index()

In [ ]:
# Solução alternativa mais segura porque o merge acontece de maneira distribuída no cluster
# Assim, somente o resultado final que terá 10 linhas será enviado para o client

df = movies_df[['movieId', 'title']].merge(
    df.compute().reset_index(),
    on='movieId',
    how='inner'
).sort_values(by='ratings_count', ascending=False)\
  .compute()\
  .set_index('movieId') # O set_index do Dask Dataframe ordena os dados por padrão, por isso usei o do pandas
df

## Atividade Turma

### Quais são os top 10 filmes com maior total da soma das avaliações?